# PhysREVE — Physics-Informed REVE Pretraining

**What this notebook implements:**  
A REVE-scale EEG foundation model extended with four physics-grounded components,  
all derived from the biophysical equation every EEG FM currently ignores:

$$\mathbf{y} = \mathbf{L} \cdot \mathbf{s} + \varepsilon$$

| Component | Where it lives | What it adds |
|-----------|---------------|---------------|
| **4D pos enc** (x,y,z,t) | Patch embedding | Geometric electrode identity — *inherited from REVE* |
| **Leadfield attention bias** | Spatial head | Physics prior: which channels share sources |
| **Source decoder + $\mathcal{L}_{phys}$** | Pretraining loss | $\mathbf{L}\hat{\mathbf{s}} \approx \mathbf{y}$ consistency |
| **SNR alignment $\mathcal{L}_{snr}$** | Pretraining loss | Learn from signal, not artifact — *from EEGPT* |

During **fine-tuning on labeled BCI data**, the asymmetry loss is added:

| Component | Where it lives | What it adds |
|-----------|---------------|---------------|
| **Hemispheric asymmetry $\mathcal{L}_{asym}$** | Fine-tuning loss | ERD in contralateral motor cortex |

---
## Relationship to REVE

```
REVE (NeurIPS 2025)              PhysREVE (this notebook)
─────────────────────────        ────────────────────────────────────────
4D pos enc (x,y,z,t)         →  4D pos enc  [UNCHANGED]
Spatio-temporal Transformer  →  + Leadfield attention bias  [+physics]
MAE objective                →  + Source decoder + L_phys + L_snr  [+physics]
60k hrs / 92 datasets        →  Same corpus  [UNCHANGED]
Sensor-space embeddings      →  Physics-grounded embeddings
Linear probe SOTA            →  Expected improvement on MI tasks
```

---
## Notebook structure
1. Installation & imports  
2. PhysREVEConfig — all hyperparameters in one place  
3. 4D Positional Encoding  
4. Forward Model — leadfield computation (MNE)  
5. Leadfield Attention Bias  
6. PhysREVE model architecture  
7. Physics-informed pretraining losses  
8. Pretraining loop (MAE + physics)  
9. Fine-tuning on labeled BCI data (+ asymmetry loss)  
10. Evaluation & comparison vs REVE baseline  
11. Source lateralization analysis  
12. Ablation: which physics loss contributes what

In [1]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo = '/content/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q', 'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    # Colab already ships torch, numpy, scipy, sklearn, matplotlib, seaborn, tqdm, requests
    # Install only what's missing
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mne>=1.6', 'moabb>=1.0', 'xgboost'])
    print('Colab environment ready.')
else:
    print('Local environment — ensure you ran: pip install -e .')

In [3]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from dataclasses import dataclass
from pathlib import Path
from copy import deepcopy
from tqdm.auto import tqdm
import mne
from mne.datasets import fetch_fsaverage
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
print(f'MNE     : {mne.__version__}')

Device  : cuda
PyTorch : 2.10.0+cu128
MNE     : 1.12.0


## 2. PhysREVEConfig — All Hyperparameters

In [4]:
@dataclass
class PhysREVEConfig:
    """
    Central configuration for PhysREVE.

    Design choices and their rationale:
    ─────────────────────────────────────────────────────────────────────
    d_model=256     REVE-base size. Enough expressivity, fits on a single GPU.
    n_heads=8       4 for spatial attention, 4 for temporal attention
                    (relevant if using criss-cross — see Section 6 comments).
    patch_size=200  1 second @ 200 Hz. REVE's published setting.
    n_sources=1284  ico-3 source space, both hemispheres (642 × 2).
                    Reduced to ~300 if using volume sphere model.
    lambda_phys     Physics consistency loss weight. Start at 0.15,
                    anneal to 0 over last 20% of training if needed.
    lambda_snr      SNR alignment loss weight. Prevents learning artifacts.
                    Borrowed from EEGPT.
    d_pos_*         4D positional encoding: 64 dims per coordinate.
                    Must sum to d_model.
    """
    # ── Transformer ─────────────────────────────────────────────────────
    d_model      : int   = 256
    n_heads      : int   = 8
    n_layers     : int   = 12
    d_ff         : int   = 1024
    dropout      : float = 0.2      # increased: reduce overfitting (24M params, ~280 train trials)

    # ── Tokenisation ────────────────────────────────────────────────────
    patch_size   : int   = 50      # samples per patch  (1s @ 200 Hz)
    max_channels : int   = 256      # max electrode count
    max_patches  : int   = 512      # max temporal patches
    mask_ratio   : float = 0.75     # fraction of patches masked (MAE)
    block_t      : int   = 4        # contiguous block masking: temporal span
    block_c      : int   = 2        # contiguous block masking: channel span

    # ── Source space ────────────────────────────────────────────────────
    n_sources    : int   = 1284     # ico-3, both hemispheres

    # ── Physics loss weights ─────────────────────────────────────────────
    lambda_phys  : float = 0.15     # L @ source ≈ sensor
    lambda_snr   : float = 0.05     # high-SNR patch alignment
    # Fine-tuning only:
    lambda_asym  : float = 0.05     # hemispheric ERD asymmetry (reduced: tanh + delay handle stability)

    # ── Physics architecture ─────────────────────────────────────────────
    leadfield_bias_scale : float = 0.5    # α for attention bias

    # ── 4D positional encoding  (must sum to d_model) ──────────────────
    d_pos_x      : int   = 64
    d_pos_y      : int   = 64
    d_pos_z      : int   = 64
    d_pos_t      : int   = 64

    def __post_init__(self):
        total = self.d_pos_x + self.d_pos_y + self.d_pos_z + self.d_pos_t
        assert total == self.d_model, (
            f'4D pos enc dims sum to {total}, expected {self.d_model}'
        )


CFG = PhysREVEConfig()

# BCI IV 2a specifics (used in fine-tuning sections)
CH_NAMES = [
    'Fz',  'FC3', 'FC1', 'FCz', 'FC2', 'FC4',
    'C5',  'C3',  'C1',  'Cz',  'C2',  'C4',  'C6',
    'CP3', 'CP1', 'CPz', 'CP2', 'CP4',
    'P1',  'Pz',  'P2',  'POz'
]
N_CH      = len(CH_NAMES)   # 22
SFREQ     = 200.0           # Hz
N_CLASSES = 4
CLASS_NAMES = ['Left Hand', 'Right Hand', 'Feet', 'Tongue']
LABEL_MAP   = {'left_hand': 0, 'right_hand': 1, 'feet': 2, 'tongue': 3}

print(f'PhysREVEConfig OK')
print(f'  d_model={CFG.d_model}, n_heads={CFG.n_heads}, n_layers={CFG.n_layers}')
print(f'  patch_size={CFG.patch_size} samples, n_sources={CFG.n_sources}')
print(f'  λ_phys={CFG.lambda_phys}, λ_snr={CFG.lambda_snr}, λ_asym={CFG.lambda_asym}')

PhysREVEConfig OK
  d_model=256, n_heads=8, n_layers=12
  patch_size=50 samples, n_sources=1284
  λ_phys=0.15, λ_snr=0.05, λ_asym=0.05


In [5]:
# ── Experiment Tracker ──────────────────────────────────────────────────────
# Bump EXP_ID each run. All figures and result logs go into experiments/EXP_ID/
import os, json as _json
from datetime import datetime

EXP_ID   = 'EXP_004'
EXP_DESC = 'P2+P5: cross-subject pretrain (subj 2-9), dropout 0.1→0.2, label_smoothing=0.1'

EXP_DIR  = os.path.join('experiments', EXP_ID)
os.makedirs(EXP_DIR, exist_ok=True)

print(f'Experiment : {EXP_ID}')
print(f'Description: {EXP_DESC}')
print(f'Output dir : {EXP_DIR}/')
print(f'Started    : {datetime.now().strftime("%Y-%m-%d %H:%M")}')


Experiment : EXP_004
Description: P2+P5: cross-subject pretrain (subj 2-9), dropout 0.1→0.2, label_smoothing=0.1
Output dir : experiments/EXP_004/
Started    : 2026-04-07 21:21


## 3. 4D Positional Encoding

REVE's core contribution: encode every patch by its electrode geometry (x,y,z) **and** temporal position (t).
This allows the model to generalise to unseen electrode layouts — it only needs the 3D coordinates.

PhysREVE uses the same encoding but adds a **noise perturbation** on the spatial coordinates during pretraining
(σ ≈ 3mm) to simulate inter-session electrode placement variability — REVE's published trick.

**Why 64 dims per axis?**  
Each sinusoidal encoding spans frequencies from 1/10000 to 1 cycle per unit.  
64 dims gives 32 frequency components — sufficient to distinguish electrode positions
at ~1mm resolution over a ~180mm head.

In [6]:
class FourDPositionalEncoding(nn.Module):
    """
    Continuous 4D positional encoding for EEG patches.

    Each patch token (electrode e, time window t) receives a positional encoding
    built from the electrode's 3D scalp coordinates plus the temporal patch index.

    Unlike REVE's original (which uses a single MLP), we use independent
    sinusoidal encodings per axis — this makes the geometry explicit and
    interpretable: axis frequencies are orthogonal.

    Args:
        cfg: PhysREVEConfig
        coord_noise_std: σ for coordinate perturbation during training (metres)
    """
    def __init__(self, cfg: PhysREVEConfig, coord_noise_std: float = 0.003):
        super().__init__()
        self.cfg = cfg
        self.noise_std = coord_noise_std

        # Frequency bases for each axis — fixed, not learned
        def freq_base(dim):
            return 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))

        self.register_buffer('freq_x', freq_base(cfg.d_pos_x))
        self.register_buffer('freq_y', freq_base(cfg.d_pos_y))
        self.register_buffer('freq_z', freq_base(cfg.d_pos_z))
        self.register_buffer('freq_t', freq_base(cfg.d_pos_t))

    def _encode_axis(self, v: torch.Tensor, freqs: torch.Tensor, dim: int):
        """
        v:     (...)      scalar or batched scalar value
        freqs: (dim//2,)
        returns: (..., dim)
        """
        angles = v.unsqueeze(-1) * freqs               # (..., dim//2)
        enc = torch.zeros(*v.shape, dim, device=v.device)
        enc[..., 0::2] = torch.sin(angles)
        enc[..., 1::2] = torch.cos(angles)
        return enc

    def forward(
        self,
        elec_xyz: torch.Tensor,   # (B, n_ch, 3)  electrode positions in metres
        t_idx:    torch.Tensor,   # (B, n_patches) temporal patch indices
    ) -> torch.Tensor:
        """
        Returns: (B, n_ch, n_patches, d_model)  positional encoding per (ch, patch)
        """
        B, C, _ = elec_xyz.shape
        P = t_idx.shape[1]

        # Optional noise on spatial coordinates (REVE robustness trick)
        if self.training and self.noise_std > 0:
            elec_xyz = elec_xyz + torch.randn_like(elec_xyz) * self.noise_std

        # Spatial encoding: (B, n_ch, d_pos_*)
        ex = self._encode_axis(elec_xyz[..., 0], self.freq_x, self.cfg.d_pos_x)  # (B,C,dx)
        ey = self._encode_axis(elec_xyz[..., 1], self.freq_y, self.cfg.d_pos_y)  # (B,C,dy)
        ez = self._encode_axis(elec_xyz[..., 2], self.freq_z, self.cfg.d_pos_z)  # (B,C,dz)
        spatial = torch.cat([ex, ey, ez], dim=-1)          # (B, C, d_spatial)

        # Temporal encoding: (B, P, d_pos_t)
        et = self._encode_axis(t_idx.float(), self.freq_t, self.cfg.d_pos_t)     # (B,P,dt)

        # Broadcast to (B, C, P, d_model)
        spatial_bcast = spatial.unsqueeze(2).expand(B, C, P, -1)
        temporal_bcast = et.unsqueeze(1).expand(B, C, P, -1)

        return torch.cat([spatial_bcast, temporal_bcast], dim=-1)  # (B, C, P, d_model)


# ── Quick validation ─────────────────────────────────────────────────────────
_pos = FourDPositionalEncoding(CFG).to(device)
_xyz = torch.randn(2, 22, 3).to(device) * 0.05    # realistic scalp coords ~50mm
_t   = torch.arange(10).unsqueeze(0).expand(2, -1).to(device)
_enc = _pos(_xyz, _t)
print(f'4D pos enc output: {_enc.shape}')           # (2, 22, 10, 256)
assert _enc.shape == (2, 22, 10, CFG.d_model)
assert not _enc.isnan().any(), 'NaN in pos enc'
del _pos, _xyz, _t, _enc
print('FourDPositionalEncoding: OK')

4D pos enc output: torch.Size([2, 22, 10, 256])
FourDPositionalEncoding: OK


## 4. Forward Model — Leadfield Computation

We use MNE-Python to compute $\mathbf{L} \in \mathbb{R}^{C \times N}$ using a sphere conductor model.
This is **precomputed once** and stored as a fixed tensor throughout training.

**At REVE scale:** REVE trains on 92 datasets with heterogeneous montages.
To handle this, we compute a *reference* leadfield on the fsaverage template montage,
then for each dataset we either:
- Use the subset of rows corresponding to its electrodes (if they're in the template), or
- Re-compute L for that dataset's specific montage (cheap: sphere model takes <30s)

The column-normalised L is used for the physics loss.  
The row-normalised L is used for the leadfield attention bias.

In [7]:
def build_leadfield(
    ch_names: list,
    sfreq:    float = 200.0,
    grid_pos: float = 10.0,    # mm — source grid spacing
    mindist:  float = 5.0,     # mm — min source-to-surface distance
    verbose:  bool  = False,
):
    """
    Compute the EEG leadfield matrix for a given electrode configuration.

    Uses MNE's 3-layer sphere conductor model — no individual MRI needed.
    The sphere is automatically fitted to the electrode positions.

    Returns
    -------
    L_col  : np.ndarray  (n_ch, n_src)  column-normalised  — for physics loss
    L_row  : np.ndarray  (n_ch, n_src)  row-normalised     — for attention bias
    src_pos: np.ndarray  (n_src, 3)     active source positions (metres)
    info   : mne.Info
    """
    # ── Electrode positions ───────────────────────────────────────────────
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    montage = mne.channels.make_standard_montage('standard_1020')
    info.set_montage(montage, match_case=False, on_missing='warn')

    # ── Sphere conductor model ────────────────────────────────────────────
    sphere = mne.make_sphere_model('auto', 'auto', info, verbose=False)

    # ── Volume source space ───────────────────────────────────────────────
    src = mne.setup_volume_source_space(
        pos=grid_pos, sphere=sphere, mindist=mindist, verbose=False
    )
    n_sources = src[0]['nuse']
    if verbose:
        print(f'  Source space: {n_sources} active dipoles')

    # ── Forward solution ──────────────────────────────────────────────────
    fs_dir = Path(fetch_fsaverage(verbose=False)).parent
    trans  = str(fs_dir / 'fsaverage' / 'bem' / 'fsaverage-trans.fif')
    fwd    = mne.make_forward_solution(
        info, trans=trans, src=src, bem=sphere,
        meg=False, eeg=True, verbose=False
    )
    fwd_f  = mne.convert_forward_solution(fwd, force_fixed=True, verbose=False)
    L      = fwd_f['sol']['data'].astype(np.float32)   # (n_ch, n_src)

    if verbose:
        print(f'  Leadfield shape: {L.shape}')

    # ── Normalise ─────────────────────────────────────────────────────────
    # Column-normalised: each source has unit scalp projection
    col_n  = np.linalg.norm(L, axis=0, keepdims=True).clip(min=1e-8)
    L_col  = L / col_n

    # Row-normalised: each electrode sees unit-norm source mixture
    row_n  = np.linalg.norm(L, axis=1, keepdims=True).clip(min=1e-8)
    L_row  = L / row_n

    # Active source positions
    active_mask = src[0]['inuse'].astype(bool)
    src_pos = src[0]['rr'][active_mask]  # (n_src, 3)

    return L_col, L_row, src_pos, info


print('Building leadfield (sphere model, ~1 min)...')
L_col_np, L_row_np, src_pos_np, info = build_leadfield(
    CH_NAMES, sfreq=SFREQ, grid_pos=10.0, verbose=True
)

N_SRC = L_col_np.shape[1]
CFG.n_sources = N_SRC      # update config to match actual source count

# Move to device as fixed tensors
L_col = torch.tensor(L_col_np, dtype=torch.float32).to(device)  # physics loss
L_row = torch.tensor(L_row_np, dtype=torch.float32).to(device)  # attention bias

print(f'\nLeadfield ready:')
print(f'  L_col (physics loss) : {L_col.shape}')
print(f'  L_row (attn bias)    : {L_row.shape}')
print(f'  Source positions     : {src_pos_np.shape}')

# ── Get electrode positions as tensor for pos enc ────────────────────────────
elec_pos_np = np.stack([ch['loc'][:3] for ch in info['chs']], axis=0)  # (n_ch, 3)
ELEC_XYZ = torch.tensor(elec_pos_np, dtype=torch.float32).to(device)   # (n_ch, 3)

Building leadfield (sphere model, ~1 min)...
  Source space: 1496 active dipoles
  Leadfield shape: (22, 1496)

Leadfield ready:
  L_col (physics loss) : torch.Size([22, 1496])
  L_row (attn bias)    : torch.Size([22, 1496])
  Source positions     : (1496, 3)


In [8]:
# ── Motor Cortex ROI Definition ──────────────────────────────────────────────
# Needed for the fine-tuning asymmetry loss.
# C3 = left motor cortex electrode, C4 = right motor cortex electrode.

def motor_roi_indices(info, src_pos, ch_names, radius=0.030, depth=0.78):
    """
    Find source indices near C3 (left motor) and C4 (right motor).

    depth: fraction of electrode-to-centre distance → approximate cortical depth
    radius: search radius in metres (30mm default)
    """
    def elec_pos(name):
        idx = ch_names.index(name)
        return np.array(info['chs'][idx]['loc'][:3])

    c3 = elec_pos('C3') * depth
    c4 = elec_pos('C4') * depth

    d_lh = np.linalg.norm(src_pos - c3, axis=1)
    d_rh = np.linalg.norm(src_pos - c4, axis=1)

    lh = np.where(d_lh < radius)[0]
    rh = np.where(d_rh < radius)[0]

    # Fallback: take nearest 10 sources
    if len(lh) < 3: lh = np.argsort(d_lh)[:10]
    if len(rh) < 3: rh = np.argsort(d_rh)[:10]

    return lh, rh


lh_idx_np, rh_idx_np = motor_roi_indices(info, src_pos_np, CH_NAMES)
LH_IDX = torch.tensor(lh_idx_np, dtype=torch.long).to(device)
RH_IDX = torch.tensor(rh_idx_np, dtype=torch.long).to(device)
print(f'Left motor cortex : {len(LH_IDX)} sources')
print(f'Right motor cortex: {len(RH_IDX)} sources')

Left motor cortex : 84 sources
Right motor cortex: 83 sources


## 5. Leadfield Attention Bias

The core spatial novelty: instead of letting the model learn from scratch which electrodes
are geometrically related, we tell it via the leadfield.

**How it works:**  
Two electrodes have similar leadfield rows if they share many cortical sources.  
We compute electrode-electrode cosine similarity via shared source overlap:

$$\mathbf{B}_{ij} = \frac{\mathbf{L}_{i,:}}{\|\mathbf{L}_{i,:}\|} \cdot \frac{\mathbf{L}_{j,:}}{\|\mathbf{L}_{j,:}\|}$$

This matrix $\mathbf{B}$ is added to the spatial attention logits as a **fixed bias**
(analogous to ALiBi in language models, but grounded in physics rather than token distance):

$$\text{Attn}(i,j) = \frac{(\mathbf{q}_i \cdot \mathbf{k}_j)}{\sqrt{d}} + \alpha \cdot \mathbf{B}_{ij}$$

In a criss-cross architecture, this bias only applies to the **spatial head** — exactly right.
In REVE's joint attention, it applies whenever two tokens are from different channels.

In [9]:
def compute_leadfield_bias(L_row: torch.Tensor) -> torch.Tensor:
    """
    Compute the electrode-electrode physics similarity matrix.

    L_row: (n_ch, n_src)  row-normalised leadfield
    Returns: (n_ch, n_ch)  cosine similarity  ∈ [-1, 1]
    """
    # L_row is already row-normalised: each row has unit norm
    return L_row @ L_row.T   # (n_ch, n_ch)


class LeadfieldAttentionBias(nn.Module):
    """
    Adds a fixed, physics-grounded bias to the spatial attention scores.

    This module:
    1. Holds the precomputed electrode-electrode similarity matrix
    2. Expands it to match the attention head / patch structure
    3. Scales by a learnable scalar α (initialised to cfg.leadfield_bias_scale)

    The bias is FIXED (no gradient through L) — it is a physics prior,
    not a learned parameter. Only α is learned.
    """
    def __init__(self, L_row: torch.Tensor, cfg: PhysREVEConfig):
        super().__init__()
        bias = compute_leadfield_bias(L_row)  # (n_ch, n_ch)
        self.register_buffer('bias', bias)     # fixed — no grad
        # Learnable scale — let the model decide how much to trust physics
        self.alpha = nn.Parameter(
            torch.tensor(cfg.leadfield_bias_scale)
        )

    def forward(
        self,
        attn_logits:  torch.Tensor,   # (B*H, n_tok, n_tok)
        ch_tok_mask:  torch.Tensor,   # (n_tok, n_tok) bool — True where both tokens are channels
        ch_indices:   torch.Tensor,   # (n_tok,) int — which channel each token belongs to (-1 if CLS)
    ) -> torch.Tensor:
        """
        Add physics bias to attention logits for channel-to-channel pairs.

        Tokens that are not from the same channel pair (e.g. temporal tokens
        or CLS token) are not affected.
        """
        n_tok = attn_logits.shape[-1]
        # Build (n_tok, n_tok) bias matrix from electrode similarity
        # Only for channel pairs (ignore CLS and cross-temporal pairs)
        bias_mat = torch.zeros(n_tok, n_tok, device=attn_logits.device)
        valid = ch_tok_mask                          # (n_tok, n_tok) bool
        if valid.any():
            i_idx = ch_indices.clamp(min=0)          # replace -1 (CLS) with 0
            full_bias = self.bias[i_idx][:, i_idx]   # (n_tok, n_tok)
            bias_mat[valid] = full_bias[valid]

        return attn_logits + self.alpha * bias_mat.unsqueeze(0)


# ── Validate ──────────────────────────────────────────────────────────────────
_lb = LeadfieldAttentionBias(L_row, CFG)
bias = _lb.bias
assert bias.shape == (N_CH, N_CH)
# Diagonal = 1.0 (electrode with itself)
assert (bias.diagonal() - 1.0).abs().max() < 0.01, 'Diagonal should be ~1.0'
print(f'Leadfield bias matrix: {bias.shape}')
print(f'  Diagonal (self-similarity): {bias.diagonal().mean():.3f}  (expected 1.0)')
print(f'  Off-diagonal mean: {bias.fill_diagonal_(0).mean():.3f}')
del _lb

Leadfield bias matrix: torch.Size([22, 22])
  Diagonal (self-similarity): 1.000  (expected 1.0)
  Off-diagonal mean: 0.479


## 6. PhysREVE Model Architecture

In [10]:
class PhysREVEPatchEmbed(nn.Module):
    """
    Tokenise raw EEG into (channel, time-patch) tokens.

    Pipeline:
      raw EEG (B, C, T)
        → depthwise temporal conv → patch features  (B, C, P, d_model)
        → add 4D positional encoding
        → flatten to token sequence  (B, C*P, d_model)

    The depthwise conv (groups=1 here, but can be C for true depthwise)
    extracts local temporal features within each channel independently.
    """
    def __init__(self, cfg: PhysREVEConfig):
        super().__init__()
        self.patch_size = cfg.patch_size
        self.d = cfg.d_model

        # Channel-wise spatial filter (like a learnable CSP, 1 filter per channel)
        self.spatial = nn.Conv1d(1, cfg.d_model, kernel_size=1, bias=False)

        # Temporal patch embedding
        self.temporal = nn.Sequential(
            nn.Conv1d(cfg.d_model, cfg.d_model,
                      kernel_size=cfg.patch_size,
                      stride=cfg.patch_size, bias=False),
            nn.GELU()
        )
        self.norm = nn.LayerNorm(cfg.d_model)
        self.pos_enc = FourDPositionalEncoding(cfg)

    def forward(self, x: torch.Tensor, elec_xyz: torch.Tensor):
        """
        x:        (B, C, T)   raw EEG, z-scored
        elec_xyz: (B, C, 3)   electrode positions in metres

        Returns:
          tokens:        (B, C*P, d)  flattened token sequence
          sensor_patches:(B, C, P)    raw signal averaged per patch (for physics loss)
          ch_indices:    (C*P,)       which channel each token belongs to
        """
        B, C, T = x.shape
        P = T // self.patch_size

        # ── Raw sensor patches (target for physics loss) ───────────────────
        xp = x[:, :, :P * self.patch_size]
        xp = xp.reshape(B, C, P, self.patch_size).mean(-1)  # (B, C, P)

        # ── Patch embedding — process each channel independently ───────────
        # Reshape to treat each channel as a 1-channel signal
        x_ch = x.reshape(B * C, 1, T)                       # (B*C, 1, T)
        h    = self.spatial(x_ch)                            # (B*C, d, T)
        h    = self.temporal(h)                              # (B*C, d, P)
        h    = h.transpose(1, 2)                             # (B*C, P, d)
        h    = self.norm(h)
        h    = h.reshape(B, C, P, self.d)                    # (B, C, P, d)

        # ── 4D positional encoding ─────────────────────────────────────────
        t_idx = torch.arange(P, device=x.device).unsqueeze(0).expand(B, -1)  # (B, P)
        pe    = self.pos_enc(elec_xyz, t_idx)                # (B, C, P, d)
        h     = h + pe

        # ── Flatten (C, P) → tokens ────────────────────────────────────────
        tokens = h.reshape(B, C * P, self.d)                 # (B, C*P, d)

        # Channel index per token (for leadfield bias routing)
        ch_indices = torch.arange(C, device=x.device).unsqueeze(1)\
                         .expand(C, P).reshape(-1)           # (C*P,)

        return tokens, xp, ch_indices


class PhysREVETransformerLayer(nn.Module):
    """
    Transformer encoder layer with optional leadfield attention bias.

    Note on criss-cross extension:
    To implement CBraMod-style criss-cross, replace the single MHA with
    two parallel heads — one attending only across channels (spatial)
    and one across time (temporal). The leadfield bias applies to the
    spatial head only. This is left as an extension exercise.
    """
    def __init__(self, cfg: PhysREVEConfig, lf_bias: LeadfieldAttentionBias):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            cfg.d_model, cfg.n_heads,
            dropout=cfg.dropout, batch_first=True
        )
        self.ff = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
            nn.Dropout(cfg.dropout)
        )
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.lf_bias = lf_bias

    def forward(
        self,
        x:           torch.Tensor,  # (B, n_tok, d)
        ch_indices:  torch.Tensor,  # (n_tok,) int  — channel per token
        attn_mask:   torch.Tensor = None,
    ):
        # Pre-LN (more stable than post-LN for large models)
        x_n = self.norm1(x)

        # Compute attention with leadfield bias ──────────────────────────
        # MHA needs attn_bias as (n_tok, n_tok) or None
        # We build a simple channel-pair mask and delegate to lf_bias
        n_tok = x.shape[1]
        ch_mask = (ch_indices.unsqueeze(0) >= 0) & (ch_indices.unsqueeze(1) >= 0)
        # ch_mask: (n_tok, n_tok) — True for channel-to-channel pairs

        # Standard MHA first
        attn_out, _ = self.attn(x_n, x_n, x_n, need_weights=False)

        # Add leadfield bias directly to the residual stream
        # (Approximate: adding to residual rather than to logits for simplicity.
        # For exact logit-level bias, replace with custom attention implementation.)
        # ── Physics-aware residual correction ────────────────────────────
        lf_correction = self.lf_bias.alpha * (
            self.lf_bias.bias[ch_indices.clamp(0)][:, ch_indices.clamp(0)].unsqueeze(0)
            * ch_mask.float().unsqueeze(0)
        ).mean(-1, keepdim=True).expand_as(x_n)
        # lf_correction is small and acts as a channel-topology-aware
        # regulariser on the residual stream

        x = x + attn_out + lf_correction
        x = x + self.ff(self.norm2(x))
        return x


class PhysREVEEncoder(nn.Module):
    """
    The full PhysREVE encoder: patch embedding → Transformer → CLS embedding.

    Used for both pretraining (with source decoder attached) and fine-tuning
    (with classification head attached).
    """
    def __init__(self, cfg: PhysREVEConfig, lf_bias: LeadfieldAttentionBias):
        super().__init__()
        self.patch_embed = PhysREVEPatchEmbed(cfg)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, cfg.d_model) * 0.02)
        self.layers      = nn.ModuleList([
            PhysREVETransformerLayer(cfg, lf_bias) for _ in range(cfg.n_layers)
        ])
        self.final_norm  = nn.LayerNorm(cfg.d_model)
        self.d = cfg.d_model

    def forward(self, x: torch.Tensor, elec_xyz: torch.Tensor):
        """
        x:        (B, C, T)
        elec_xyz: (B, C, 3)
        Returns:
          cls_out:       (B, d)         — pooled representation
          patch_out:     (B, C*P, d)    — per-token representations
          sensor_patches:(B, C, P)      — raw signal per patch
          ch_indices:    (C*P+1,)       — channel index per token (+1 for CLS)
        """
        B = x.shape[0]

        tokens, sen_p, ch_idx = self.patch_embed(x, elec_xyz)  # (B, C*P, d)

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)          # (B, C*P+1, d)
        ch_idx_cls = torch.cat([
            torch.tensor([-1], device=x.device), ch_idx   # -1 = CLS
        ])

        for layer in self.layers:
            tokens = layer(tokens, ch_idx_cls)

        tokens = self.final_norm(tokens)
        return tokens[:, 0], tokens[:, 1:], sen_p, ch_idx_cls[1:]

In [11]:
class PhysREVEPretrainModel(nn.Module):
    """
    PhysREVE pretraining model.

    = PhysREVEEncoder
    + MAE decoder  (reconstruct masked sensor patches)
    + Source decoder (estimate source activations → physics loss)

    The source decoder is DISCARDED after pretraining.
    The encoder is transferred to fine-tuning.
    """
    def __init__(self, cfg: PhysREVEConfig, lf_bias: LeadfieldAttentionBias):
        super().__init__()
        d = cfg.d_model

        self.encoder = PhysREVEEncoder(cfg, lf_bias)

        # ── MAE decoder ────────────────────────────────────────────────────
        # Lightweight: 2 Transformer layers, reconstruct to patch_size samples
        self.mask_token  = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        dec_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=4, dim_feedforward=d*2,
            dropout=0.1, batch_first=True, norm_first=True
        )
        self.mae_decoder = nn.TransformerEncoder(dec_layer, num_layers=2)
        self.mae_head    = nn.Linear(d, cfg.patch_size)    # reconstruct raw patch

        # ── Source decoder ─────────────────────────────────────────────────
        # Per-patch: estimate source activation → shape (B, n_src, P)
        # Used only during pretraining for the physics loss.
        self.src_decoder = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, cfg.n_sources)
            # No activation: dipole sources are signed
        )

    def forward(
        self,
        x:         torch.Tensor,   # (B, C, T)
        elec_xyz:  torch.Tensor,   # (B, C, 3)
        mask:      torch.Tensor,   # (B, C, P) bool — True = masked
    ):
        """
        Returns:
          mae_recon:     (B, C, P, patch_size)  reconstructed raw patches
          src_acts:      (B, n_src, P)          source activation estimates
          sensor_patches:(B, C, P)              observed EEG in patch space
          patch_enc:     (B, C*P, d)            encoded patch representations
        """
        B, C, T = x.shape
        P = T // self.encoder.patch_embed.patch_size

        # Apply mask: replace masked patches with mask token
        # We'll let the full signal through to the encoder but zero masked patches
        # (visible patches only — standard MAE approach)
        x_masked = x.clone()
        for b in range(B):
            for c in range(C):
                for p in range(P):
                    if mask[b, c, p]:
                        ps = self.encoder.patch_embed.patch_size
                        x_masked[b, c, p*ps:(p+1)*ps] = 0.0

        # Encode visible patches
        cls_out, patch_enc, sen_p, ch_idx = self.encoder(x_masked, elec_xyz)
        # patch_enc: (B, C*P, d)

        # ── MAE decoder ────────────────────────────────────────────────────
        # Replace masked token positions with learnable mask token
        patch_enc_dec = patch_enc.clone()
        mask_flat = mask.reshape(B, C * P)           # (B, C*P)
        for b in range(B):
            masked_pos = mask_flat[b].nonzero(as_tuple=True)[0]
            patch_enc_dec[b, masked_pos] = self.mask_token

        dec_out = self.mae_decoder(patch_enc_dec)    # (B, C*P, d)
        mae_recon_flat = self.mae_head(dec_out)      # (B, C*P, patch_size)
        mae_recon = mae_recon_flat.reshape(B, C, P, -1)  # (B, C, P, patch_size)

        # ── Source decoder ─────────────────────────────────────────────────
        # Apply to all patch tokens: (B, C*P, d) → (B, C*P, n_src)
        src_all = self.src_decoder(patch_enc)        # (B, C*P, n_src)
        # Average source estimates across channels per time patch
        # Reshape: (B, C, P, n_src) → mean over C → (B, P, n_src)
        src_all = src_all.reshape(B, C, P, -1)
        src_avg = src_all.mean(dim=1)                # (B, P, n_src)
        src_acts = src_avg.transpose(1, 2)           # (B, n_src, P)

        return mae_recon, src_acts, sen_p, patch_enc


class PhysREVEFineTuneModel(nn.Module):
    """
    PhysREVE fine-tuning model.

    = Pretrained PhysREVEEncoder (loaded from checkpoint)
    + Source decoder (retained for asymmetry loss during fine-tuning)
    + Classification head

    After fine-tuning, only the encoder + classifier are needed for inference.
    """
    def __init__(self, cfg: PhysREVEConfig, pretrain_model: PhysREVEPretrainModel):
        super().__init__()
        self.encoder     = pretrain_model.encoder
        self.src_decoder = pretrain_model.src_decoder    # for asymmetry loss
        self.classifier  = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, N_CLASSES)
        )
        self._init_head()

    def _init_head(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor, elec_xyz: torch.Tensor):
        cls_out, patch_enc, sen_p, ch_idx = self.encoder(x, elec_xyz)

        # Classification
        logits = self.classifier(cls_out)            # (B, n_classes)

        # Source estimates (for asymmetry loss during fine-tuning)
        B, C, T = x.shape
        P = T // self.encoder.patch_embed.patch_size
        src_all = self.src_decoder(patch_enc)        # (B, C*P, n_src)
        src_all = src_all.reshape(B, C, P, -1).mean(1)  # (B, P, n_src)
        src_acts = src_all.transpose(1, 2)           # (B, n_src, P)

        return logits, src_acts, sen_p


# ── Architecture validation ───────────────────────────────────────────────────
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

_lf_bias = LeadfieldAttentionBias(L_row, CFG).to(device)
_model   = PhysREVEPretrainModel(CFG, _lf_bias).to(device)

B_test, C_test = 2, N_CH
T_test = CFG.patch_size * 10    # 10 patches
_x    = torch.randn(B_test, C_test, T_test).to(device)
_xyz  = ELEC_XYZ.unsqueeze(0).expand(B_test, -1, -1)
_mask = torch.zeros(B_test, C_test, 10, dtype=torch.bool).to(device)
_mask[:, :, 5:] = True   # mask last 5 patches

_recon, _src, _sen, _enc = _model(_x, _xyz, _mask)

print('PhysREVEPretrainModel architecture validated:')
print(f'  Parameters       : {count_params(_model):,}')
print(f'  MAE recon        : {_recon.shape}  (B, C, P, patch_size)')
print(f'  Source activations: {_src.shape}   (B, n_src, P)')
print(f'  Sensor patches   : {_sen.shape}    (B, C, P)')
print(f'  Patch encodings  : {_enc.shape}   (B, C*P, d)')
assert not _recon.isnan().any(), 'NaN in MAE reconstruction'
assert not _src.isnan().any(),   'NaN in source activations'
del _model, _lf_bias, _x, _xyz, _mask, _recon, _src, _sen, _enc

PhysREVEPretrainModel architecture validated:
  Parameters       : 14,273,547
  MAE recon        : torch.Size([2, 22, 10, 50])  (B, C, P, patch_size)
  Source activations: torch.Size([2, 1496, 10])   (B, n_src, P)
  Sensor patches   : torch.Size([2, 22, 10])    (B, C, P)
  Patch encodings  : torch.Size([2, 220, 256])   (B, C*P, d)


## 7. Physics-Informed Pretraining Losses

Four loss terms used during pretraining:

$$\mathcal{L}_{pretrain} = \mathcal{L}_{MAE} + \lambda_p \mathcal{L}_{phys} + \lambda_s \mathcal{L}_{snr}$$

During fine-tuning on labeled BCI data:

$$\mathcal{L}_{finetune} = \mathcal{L}_{CE} + \lambda_p \mathcal{L}_{phys} + \lambda_a \mathcal{L}_{asym}$$

**$\mathcal{L}_{MAE}$** — reconstruct masked patches (standard REVE objective)  
**$\mathcal{L}_{phys}$** — $\|\text{norm}(\mathbf{L}\hat{\mathbf{s}}) - \text{norm}(\mathbf{y}_{patches})\|^2$  
**$\mathcal{L}_{snr}$** — InfoNCE: pull together high-SNR patch representations (from EEGPT)  
**$\mathcal{L}_{asym}$** — contralateral ERD: encourages correct hemispheric lateralization (fine-tuning only)

In [12]:
def mae_loss(
    recon:  torch.Tensor,    # (B, C, P, patch_size)
    target: torch.Tensor,    # (B, C, T)  raw EEG
    mask:   torch.Tensor,    # (B, C, P) bool
) -> torch.Tensor:
    """
    REVE-style MAE loss: MSE on masked patches only.
    Targets are z-scored within each patch for stability.
    """
    B, C, P, ps = recon.shape
    # Build raw targets in patch format
    t_patches = target[:, :, :P*ps].reshape(B, C, P, ps)   # (B, C, P, ps)
    # Normalise targets per patch (like REVE's patch normalisation)
    t_mu  = t_patches.mean(dim=-1, keepdim=True)
    t_std = t_patches.std( dim=-1, keepdim=True).clamp(min=1e-6)
    t_n   = (t_patches - t_mu) / t_std

    # Loss only on masked patches
    if mask.sum() == 0:
        return recon.new_zeros([])
    loss = F.mse_loss(recon[mask], t_n[mask])
    return loss


def physics_loss(
    L:           torch.Tensor,   # (n_ch, n_src)  column-normalised
    src_acts:    torch.Tensor,   # (B, n_src, P)
    sensor_p:    torch.Tensor,   # (B, n_ch, P)
    eps:  float = 1e-6,
) -> torch.Tensor:
    """
    Forward model consistency: L @ source_est ≈ sensor.

    Both sides are normalised to remove amplitude ambiguity — we enforce
    topographic SHAPE consistency, not absolute amplitude.

    Gradient flows through src_acts only (L is fixed physics).
    """
    # (n_ch, n_src) × (B, n_src, P) → (B, n_ch, P)
    recon = torch.einsum('cs,bsp->bcp', L, src_acts)

    r_n = recon   / (recon.norm(  dim=1, keepdim=True) + eps)
    y_n = sensor_p / (sensor_p.norm(dim=1, keepdim=True) + eps)

    return F.mse_loss(r_n, y_n)


def snr_alignment_loss(
    patch_enc:    torch.Tensor,   # (B, C*P, d)  patch token encodings
    x_raw:        torch.Tensor,   # (B, C, T)    raw EEG
    patch_size:   int,
    sfreq:        float = 200.0,
    neural_bands: list  = [(4,8),(8,13),(13,30)],
    snr_quantile: float = 0.5,
    temperature:  float = 0.1,
    eps:          float = 1e-8,
) -> torch.Tensor:
    """
    EEGPT-inspired SNR alignment loss.

    High-SNR patches (predominantly neural rather than artifact) are pulled
    together in the representation space using a contrastive (InfoNCE) objective.

    This discourages the model from learning to reconstruct artifacts —
    the single most common failure mode of MAE pretraining on real EEG.

    High SNR is estimated as: neural_band_power / broadband_power.
    Patches above the median SNR are treated as positive pairs.
    """
    B, C, T = x_raw.shape
    P = T // patch_size

    # ── Estimate SNR per patch ─────────────────────────────────────────────
    xp = x_raw[:, :, :P*patch_size].reshape(B, C, P, patch_size)
    # FFT along patch axis
    with torch.no_grad():
        spec = torch.fft.rfft(xp, dim=-1).abs().pow(2)     # (B, C, P, F)
        freqs = torch.fft.rfftfreq(patch_size, d=1.0/sfreq).to(x_raw.device)

        neural_mask = torch.zeros(len(freqs), dtype=torch.bool, device=x_raw.device)
        for lo, hi in neural_bands:
            neural_mask |= (freqs >= lo) & (freqs <= hi)

        neural_pwr = spec[..., neural_mask].mean(dim=(1, 3))   # (B, P)
        total_pwr  = spec.mean(dim=(1, 3)).clamp(min=eps)      # (B, P)
        snr        = neural_pwr / total_pwr                     # (B, P)

        # High-SNR mask: above median per batch
        thresh = snr.median(dim=1, keepdim=True).values
        hi_mask = snr >= thresh                                 # (B, P)

    # ── InfoNCE on high-SNR patch representations ──────────────────────────
    # Average channel representations per time patch
    z = patch_enc.reshape(B, C, P, -1).mean(dim=1)             # (B, P, d)
    z_n = F.normalize(z, dim=-1)                                # (B, P, d)

    total_loss = z.new_zeros([])
    n_terms = 0

    for b in range(B):
        hi_idx = hi_mask[b].nonzero(as_tuple=True)[0]          # indices of high-SNR patches
        K = len(hi_idx)
        if K < 2:
            continue
        z_hi = z_n[b, hi_idx]                                   # (K, d)
        sim  = z_hi @ z_hi.T / temperature                     # (K, K)
        # For each patch, treat the next patch as the positive pair
        for i in range(K - 1):
            numerator   = sim[i, i+1].exp()
            denominator = sim[i].exp().sum() - sim[i, i].exp() + eps
            total_loss  = total_loss - torch.log(numerator / denominator + eps)
            n_terms    += 1

    return total_loss / max(n_terms, 1)


def asymmetry_loss(
    src_acts: torch.Tensor,   # (B, n_src, P)
    labels:   torch.Tensor,   # (B,)  0=left, 1=right, 2=feet, 3=tongue
    lh_idx:   torch.Tensor,   # LongTensor — left motor cortex source indices
    rh_idx:   torch.Tensor,   # LongTensor — right motor cortex source indices
    eps:      float = 1e-6,
) -> torch.Tensor:
    """
    Hemispheric ERD asymmetry constraint.

    During motor imagery, the CONTRALATERAL hemisphere desynchronises (ERD):
      Left hand  (label=0) → right motor cortex ERD → rh_power < lh_power
      Right hand (label=1) → left motor cortex ERD  → lh_power < rh_power

    We encode this as: maximise (expected_sign × asymmetry).
    Only applied to left/right MI trials.
    """
    lh_p = src_acts[:, lh_idx, :].pow(2).mean(dim=(1, 2))   # (B,)
    rh_p = src_acts[:, rh_idx, :].pow(2).mean(dim=(1, 2))   # (B,)

    # Normalised asymmetry — tanh-clamped to prevent gradient collapse.
    # Raw (rh-lh)/(rh+lh) saturates hard at ±1; tanh with scale=3 gives
    # near-identical range but smooth, non-zero gradients everywhere.
    raw  = (rh_p - lh_p) / (lh_p + rh_p + eps)              # (B,) in (-1,1)
    asym = torch.tanh(3.0 * raw)                             # still in (-1,1), smoother

    mi_mask = (labels == 0) | (labels == 1)
    if mi_mask.sum() == 0:
        return src_acts.new_zeros([])

    # Left hand → negative asymmetry (right low); Right hand → positive
    sign = torch.where(labels[mi_mask] == 1,
                       asym.new_ones(mi_mask.sum()),
                       -asym.new_ones(mi_mask.sum()))
    return -torch.mean(sign * asym[mi_mask])


# ── Combined loss functions ───────────────────────────────────────────────────

def pretrain_losses(recon, src_acts, sen_p, patch_enc, x_raw, mask, L, cfg):
    """
    Pretraining total loss:
        L_MAE + λ_phys * L_phys + λ_snr * L_snr
    """
    lmae  = mae_loss(recon, x_raw, mask)
    lphys = physics_loss(L, src_acts, sen_p)
    lsnr  = snr_alignment_loss(patch_enc, x_raw, cfg.patch_size, sfreq=SFREQ)

    total = lmae + cfg.lambda_phys * lphys + cfg.lambda_snr * lsnr
    return total, {
        'mae':   lmae.item(),
        'phys':  lphys.item(),
        'snr':   lsnr.item(),
        'total': total.item()
    }


def finetune_losses(logits, src_acts, sen_p, labels, L, cfg,
                    lh_idx, rh_idx, epoch: int = 1):
    """
    Fine-tuning total loss:
        L_CE + λ_phys * L_phys + λ_asym * L_asym

    L_asym is zeroed for the first 10 epochs so CE can stabilise before
    the asymmetry constraint kicks in (avoids early collapse).
    """
    lce   = F.cross_entropy(logits, labels, label_smoothing=0.1)
    lphys = physics_loss(L, src_acts, sen_p)
    lasym = asymmetry_loss(src_acts, labels, lh_idx, rh_idx)

    # Delayed asymmetry: ramp in after epoch 10
    asym_w = cfg.lambda_asym if epoch >= 10 else 0.0
    total = lce + cfg.lambda_phys * lphys + asym_w * lasym
    return total, {
        'ce':    lce.item(),
        'phys':  lphys.item(),
        'asym':  lasym.item(),
        'total': total.item()
    }


print('All loss functions defined.')
print(f'Pretraining: L_MAE + {CFG.lambda_phys}*L_phys + {CFG.lambda_snr}*L_snr')
print(f'Fine-tuning: L_CE  + {CFG.lambda_phys}*L_phys + {CFG.lambda_asym}*L_asym')

All loss functions defined.
Pretraining: L_MAE + 0.15*L_phys + 0.05*L_snr
Fine-tuning: L_CE  + 0.15*L_phys + 0.05*L_asym


## 8. Contiguous Block Masking

REVE's key masking innovation: instead of independently random patch masking,
mask **contiguous spatio-temporal blocks** — a rectangle of (block_c × block_t) patches.

Why this is harder than random masking:
- You cannot interpolate from immediately adjacent patches
- Forces the model to learn long-range spatial AND temporal dependencies
- Analogous to image MAE block masking (He et al., 2022) but in 2D (channel × time)

In [13]:
def block_mask(
    B: int, C: int, P: int,
    ratio:   float = 0.75,
    block_t: int   = 4,
    block_c: int   = 2,
    device:  torch.device = torch.device('cpu'),
) -> torch.Tensor:
    """
    REVE-style spatio-temporal contiguous block masking.

    Randomly places (block_c × block_t) blocks of True in a (C, P) grid
    until the target mask ratio is reached.

    Returns: (B, C, P) bool  — True = masked (must be reconstructed)
    """
    mask   = torch.zeros(B, C, P, dtype=torch.bool, device=device)
    target = int(C * P * ratio)

    for b in range(B):
        n_masked = 0
        max_iter = 5000
        for _ in range(max_iter):
            if n_masked >= target:
                break
            c0 = torch.randint(0, C, ()).item()
            t0 = torch.randint(0, P, ()).item()
            c1 = min(c0 + block_c, C)
            t1 = min(t0 + block_t, P)
            mask[b, c0:c1, t0:t1] = True
            n_masked = mask[b].sum().item()

    return mask


# Validate
_m = block_mask(4, N_CH, 10, ratio=0.75, device=device)
print(f'Block mask: {_m.shape}  masked_frac={_m.float().mean():.2f}  (target 0.75)')
assert 0.5 < _m.float().mean().item() < 0.99, 'Mask ratio out of expected range'
del _m

Block mask: torch.Size([4, 22, 10])  masked_frac=0.76  (target 0.75)


## 9. Pretraining Loop

**Scale note:**  
Full REVE pretraining uses 60k hours on 92 datasets. Here we show the loop  
on BCI IV 2a as a **proof of concept** — in production, replace the dataloader  
with a multi-dataset iterator over REVE's full corpus.

The architecture, losses, and masking are identical regardless of scale.

In [14]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

# ── Load data ─────────────────────────────────────────────────────────────────
# P2 fix (BUG-03): pretraining now uses subjects 2-9 (held-out from fine-tuning).
# Subject 1 is used exclusively for fine-tuning / val / test.
# This breaks the train-pretrain overlap that was inflating fine-tuning ease.
print('Loading BCI IV 2a...')
dataset  = BNCI2014_001()
paradigm = MotorImagery(
    events=['left_hand', 'right_hand', 'feet', 'tongue'],
    n_classes=4, fmin=0.5, fmax=40.0, tmin=0.5, tmax=2.5
)

# ── Pretraining data: subjects 2-9 (no labels needed) ────────────────────────
print('  Loading pretraining subjects (2-9)...')
X_pretrain_parts = []
for subj in range(2, 10):
    X_s, _, _ = paradigm.get_data(dataset, subjects=[subj], return_epochs=False)
    P_s = X_s.shape[-1] // CFG.patch_size
    X_s = X_s[:, :, :P_s * CFG.patch_size].astype(np.float32)
    X_pretrain_parts.append(X_s)
    print(f'    Subject {subj}: {X_s.shape[0]} trials')
X_pretrain = np.concatenate(X_pretrain_parts, axis=0)
print(f'  Pretraining pool: {X_pretrain.shape[0]} trials from 8 subjects')

# ── Fine-tuning data: subject 1 only ─────────────────────────────────────────
print('  Loading fine-tuning subject (1)...')
X_s1, y_str_s1, _ = paradigm.get_data(dataset, subjects=[1], return_epochs=False)
T_WIN = X_s1.shape[-1]
P     = T_WIN // CFG.patch_size
X_s1  = X_s1[:, :, :P * CFG.patch_size].astype(np.float32)
y_all = np.array([LABEL_MAP[yi] for yi in y_str_s1])
print(f'  Subject 1: {X_s1.shape[0]} trials  ({P} patches per trial)')


class UnlabeledEEGDataset(Dataset):
    """For pretraining — returns only the signal, no labels."""
    def __init__(self, X):
        mu  = X.mean(axis=-1, keepdims=True)
        std = X.std( axis=-1, keepdims=True).clip(min=1e-8)
        self.X = torch.tensor((X - mu) / std, dtype=torch.float32)
    def __len__(self):        return len(self.X)
    def __getitem__(self, i): return self.X[i]


class LabeledEEGDataset(Dataset):
    """For fine-tuning — returns signal + label."""
    def __init__(self, X, y):
        mu  = X.mean(axis=-1, keepdims=True)
        std = X.std( axis=-1, keepdims=True).clip(min=1e-8)
        self.X = torch.tensor((X - mu) / std, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):        return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


# ── Pretrain split: all of subjects 2-9 ──────────────────────────────────────
pretrain_ds = UnlabeledEEGDataset(X_pretrain)

# ── Fine-tune split: subject 1, 70% train | 15% val | 15% test ───────────────
n     = len(X_s1)
n_tr, n_v = int(0.70*n), int(0.15*n)
idx   = np.random.default_rng(SEED).permutation(n)
idx_tr, idx_v, idx_te = idx[:n_tr], idx[n_tr:n_tr+n_v], idx[n_tr+n_v:]

finetune_ds = LabeledEEGDataset(X_s1[idx_tr], y_all[idx_tr])
val_ds      = LabeledEEGDataset(X_s1[idx_v],  y_all[idx_v])
test_ds     = LabeledEEGDataset(X_s1[idx_te], y_all[idx_te])

BATCH = 16
pretrain_loader = DataLoader(pretrain_ds,  batch_size=BATCH, shuffle=True,  drop_last=True)
finetune_loader = DataLoader(finetune_ds,  batch_size=BATCH, shuffle=True,  drop_last=True)
val_loader      = DataLoader(val_ds,       batch_size=BATCH, shuffle=False)
test_loader     = DataLoader(test_ds,      batch_size=BATCH, shuffle=False)

print(f'Pretrain : {len(pretrain_ds)} trials (subjects 2-9)')
print(f'Finetune : {len(finetune_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}  (subject 1)')

# Broadcast electrode positions across batch
def get_elec_xyz(batch_size):
    return ELEC_XYZ.unsqueeze(0).expand(batch_size, -1, -1)  # (B, C, 3)


Loading BCI IV 2a...
  Loading pretraining subjects (2-9)...
    Subject 2: 576 trials
    Subject 3: 576 trials
    Subject 4: 576 trials
    Subject 5: 576 trials
    Subject 6: 576 trials
    Subject 7: 576 trials
    Subject 8: 576 trials
    Subject 9: 576 trials
  Pretraining pool: 4608 trials from 8 subjects
  Loading fine-tuning subject (1)...
  Subject 1: 576 trials  (10 patches per trial)
Pretrain : 4608 trials (subjects 2-9)
Finetune : 403 | Val: 86 | Test: 87  (subject 1)


In [ ]:
def run_pretraining(
    cfg: PhysREVEConfig,
    loader: DataLoader,
    n_epochs: int  = 30,
    lr:       float = 3e-4,
    wd:       float = 1e-4,
    warmup_epochs: int = 5,
):
    """
    Pretrain PhysREVE on unlabeled EEG.

    In full-scale training (REVE-style):
      - Replace loader with a multi-dataset iterator
      - Scale to 60k hours, 92 datasets
      - Use gradient accumulation to simulate larger batch sizes
      - Add cosine annealing with warmup (AdamW + LAMB)

    Returns the pretrained model and training history.
    """
    lf_bias = LeadfieldAttentionBias(L_row, cfg).to(device)
    model   = PhysREVEPretrainModel(cfg, lf_bias).to(device)

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    # Cosine schedule with linear warmup
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        progress = (epoch - warmup_epochs) / max(n_epochs - warmup_epochs, 1)
        return 0.5 * (1 + np.cos(np.pi * progress))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    hist = {k: [] for k in ('mae', 'phys', 'snr', 'total')}

    print(f'Pretraining PhysREVE — {count_params(model):,} parameters')
    print(f'Epochs: {n_epochs}  |  Batches/epoch: {len(loader)}')
    print(f'Losses: MAE + {cfg.lambda_phys}·L_phys + {cfg.lambda_snr}·L_snr')

    for epoch in range(1, n_epochs + 1):
        model.train()
        ep_parts = {k: 0.0 for k in hist}
        n = 0

        for Xb in loader:
            Xb  = Xb.to(device)
            bs  = Xb.shape[0]
            xyz = get_elec_xyz(bs)

            # Sample a contiguous block mask
            msk = block_mask(
                bs, N_CH, P,
                ratio=cfg.mask_ratio,
                block_t=cfg.block_t,
                block_c=cfg.block_c,
                device=device
            )

            recon, src_acts, sen_p, patch_enc = model(Xb, xyz, msk)
            loss, parts = pretrain_losses(
                recon, src_acts, sen_p, patch_enc, Xb, msk, L_col, cfg
            )

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            for k, v in parts.items():
                ep_parts[k] += v * bs
            n += bs

        sched.step()
        for k in hist:
            hist[k].append(ep_parts[k] / n)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Ep {epoch:3d}/{n_epochs}  '
                  f'L_mae={hist["mae"][-1]:.4f}  '
                  f'L_phys={hist["phys"][-1]:.4f}  '
                  f'L_snr={hist["snr"][-1]:.4f}  '
                  f'L_total={hist["total"][-1]:.4f}  '
                  f'lr={sched.get_last_lr()[0]:.2e}')

    return model, hist


# ── Run pretraining ────────────────────────────────────────────────────────────
print('='*65)
print('PHASE 1: Pretraining (MAE + physics losses)')
print('='*65)
pretrained_model, pretrain_hist = run_pretraining(
    CFG, pretrain_loader, n_epochs=30, lr=3e-4
)

# Save encoder weights
torch.save(pretrained_model.state_dict(), f'physreve_pretrained{EXP_ID}.pt')
print(f'\nPretrained weights saved: physreve_pretrained{EXP_ID}.pt')

PHASE 1: Pretraining (MAE + physics losses)
Pretraining PhysREVE — 14,273,547 parameters
Epochs: 30  |  Batches/epoch: 288
Losses: MAE + 0.15·L_phys + 0.05·L_snr
  Ep   1/30  L_mae=1.1132  L_phys=0.0920  L_snr=1.4860  L_total=1.2013  lr=6.00e-05


## 10. Fine-Tuning on Labeled BCI Data

In [ ]:
def run_finetuning(
    pretrained: PhysREVEPretrainModel,
    cfg:        PhysREVEConfig,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    n_epochs:  int   = 50,
    lr_head:   float = 1e-3,    # higher LR for new classification head
    lr_enc:    float = 1e-4,    # lower LR for pretrained encoder
    wd:              float = 1e-4,
    freeze_enc_epochs: int   = 5,     # keep encoder frozen for first N epochs
    lambda_phys_ft:  float = 0.0,    # physics loss weight during fine-tuning.
                                      # Default 0: with ~200 trials, L_phys
                                      # competes with CE and hurts convergence.
                                      # Set to cfg.lambda_phys to replicate
                                      # the original full-physics fine-tuning.
):
    """
    Fine-tune PhysREVE on labeled BCI data.

    Uses differential learning rates:
    - Encoder (pretrained): low LR — preserve learned representations
    - Classifier head (random init): high LR — learn task quickly

    Includes the asymmetry loss to further ground motor imagery
    representations in known neuroscience.
    """
    model = PhysREVEFineTuneModel(cfg, pretrained).to(device)

    # Differential LR: encoder vs head
    enc_params  = list(model.encoder.parameters()) + list(model.src_decoder.parameters())
    head_params = list(model.classifier.parameters())
    opt = torch.optim.AdamW([
        {'params': enc_params,  'lr': lr_enc},
        {'params': head_params, 'lr': lr_head},
    ], weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr_enc/10)

    # Freeze encoder during warmup so head can orient first
    for p in enc_params:
        p.requires_grad = False

    hist = {k: [] for k in ('train_acc', 'val_acc', 'ce', 'phys', 'asym')}
    best_val = 0.0; best_state = None

    print(f'Fine-tuning PhysREVE')
    print(f'LR: encoder={lr_enc}  head={lr_head}')
    print(f'Losses: CE + {cfg.lambda_phys}·L_phys + {cfg.lambda_asym}·L_asym')

    for epoch in range(1, n_epochs + 1):
        # ── Encoder warmup: unfreeze after freeze_enc_epochs ──────────────
        if epoch == freeze_enc_epochs + 1:
            for p in enc_params:
                p.requires_grad = True
            print(f'  [Epoch {epoch}] Encoder unfrozen — physics alignment begins')

        # ── Training ──────────────────────────────────────────────────────
        model.train()
        tr_parts = {k: 0.0 for k in ('ce', 'phys', 'asym')}
        tr_acc = 0.0; n = 0

        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            xyz = get_elec_xyz(len(Xb))

            logits, src_acts, sen_p = model(Xb, xyz)
            # Temporarily override cfg.lambda_phys for fine-tuning
            _ft_cfg = type(cfg)(**{**cfg.__dict__, 'lambda_phys': lambda_phys_ft})
            loss, parts = finetune_losses(
                logits, src_acts, sen_p, yb, L_col, _ft_cfg, LH_IDX, RH_IDX,
                epoch=epoch
            )
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            bs = len(yb)
            for k in ('ce', 'phys', 'asym'):
                tr_parts[k] += parts[k] * bs
            tr_acc += (logits.argmax(1) == yb).float().sum().item()
            n += bs

        sched.step()
        tr_acc /= n

        # ── Validation ────────────────────────────────────────────────────
        model.eval()
        va_acc = 0.0; nv = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                logits, _, _ = model(Xb, get_elec_xyz(len(Xb)))
                va_acc += (logits.argmax(1) == yb).float().sum().item()
                nv += len(yb)
        va_acc /= nv

        for k in ('ce', 'phys', 'asym'):
            hist[k].append(tr_parts[k] / n)
        hist['train_acc'].append(tr_acc)
        hist['val_acc'].append(va_acc)

        if va_acc > best_val:
            best_val  = va_acc
            best_state = deepcopy(model.state_dict())

        if epoch % 10 == 0 or epoch == 1:
            print(f'  Ep {epoch:3d}/{n_epochs}  '
                  f'train={tr_acc:.3f}  val={va_acc:.3f}  '
                  f'ce={tr_parts["ce"]/n:.3f}  '
                  f'phys={tr_parts["phys"]/n:.4f}  '
                  f'asym={tr_parts["asym"]/n:.4f}')

    model.load_state_dict(best_state)
    print(f'  Best val acc: {best_val:.3f}')
    return model, hist


# ── Also train a from-scratch baseline (no pretraining) ───────────────────────
def run_baseline_finetune(cfg, train_loader, val_loader, n_epochs=50, lr=3e-4):
    """Baseline: same architecture, random init, no physics pretraining.
    
    freeze_enc_epochs=0: with random init, freezing the encoder means the head
    trains on random features for the warmup period — a bad initialisation.
    The freeze warmup is only useful when the encoder carries pretrained weights.
    """
    lf_bias = LeadfieldAttentionBias(L_row, cfg).to(device)
    scratch  = PhysREVEPretrainModel(cfg, lf_bias).to(device)  # random weights
    return run_finetuning(
        scratch, cfg, train_loader, val_loader,
        n_epochs=n_epochs, lr_head=lr, lr_enc=lr,
        freeze_enc_epochs=0,   # BUG FIX: no warmup freeze for random init
    )


print('\n' + '='*65)
print('PHASE 2a: Baseline fine-tuning (random init, no pretraining)')
print('='*65)
model_base, hist_base_ft = run_baseline_finetune(
    CFG, finetune_loader, val_loader, n_epochs=50
)

print('\n' + '='*65)
print('PHASE 2b: PhysREVE fine-tuning (from pretrained weights)')
print('='*65)
model_phys, hist_phys_ft = run_finetuning(
    pretrained_model, CFG, finetune_loader, val_loader, n_epochs=50
)

## 11. Evaluation, Ablations & Visualisation

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels, srcs = [], [], []
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits, src, _ = model(Xb, get_elec_xyz(len(Xb)))
        preds.append(logits.argmax(1).cpu())
        labels.append(yb.cpu())
        srcs.append(src.cpu())
    return (
        torch.cat(preds).numpy(),
        torch.cat(labels).numpy(),
        torch.cat(srcs)
    )

p_b, l_b, s_b = evaluate(model_base, test_loader)
p_p, l_p, s_p = evaluate(model_phys, test_loader)

acc_b = (p_b == l_b).mean()
acc_p = (p_p == l_p).mean()

print('\n' + '='*65)
print('TEST SET RESULTS')
print('='*65)
print(f'{"Model":<40} {"Accuracy":>10} {"vs Chance":>12}')
print('-'*65)
print(f'{"Baseline (random init)":<40} {acc_b*100:>9.1f}% {(acc_b-0.25)*100:>+11.1f}%')
print(f'{"PhysREVE (physics pretrained)":<40} {acc_p*100:>9.1f}% {(acc_p-0.25)*100:>+11.1f}%')
print(f'{"Chance level":<40} {25.0:>9.1f}%')
print('='*65)
print(f'Improvement from physics pretraining: {(acc_p - acc_b)*100:+.1f}%')

print('\n--- PhysREVE Classification Report ---')
print(classification_report(l_p, p_p, target_names=CLASS_NAMES))

In [ ]:
# ── Figure 1: Training analysis ────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.suptitle(
    f'PhysREVE {EXP_ID} — Physics-Informed REVE Pretraining\n'
    f'BCI Competition IV 2a · 4-class Motor Imagery',
    fontsize=13, fontweight='bold'
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── A: Pretraining loss decomposition ────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
ep = range(1, len(pretrain_hist['mae']) + 1)
ax.plot(ep, pretrain_hist['mae'],   label='L_MAE',  color='#dc2626', lw=2)
ax2 = ax.twinx()
ax2.plot(ep, pretrain_hist['phys'], label='L_phys', color='#16a34a', lw=2, ls='--')
ax2.plot(ep, pretrain_hist['snr'],  label='L_snr',  color='#9333ea', lw=2, ls=':')
ax.set(title='Pretraining Losses', xlabel='Epoch', ylabel='MAE Loss')
ax2.set_ylabel('Physics / SNR Loss')
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, fontsize=8, loc='upper right')
ax.grid(alpha=0.25)

# ── B: Fine-tuning accuracy curves ────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
ep2 = range(1, len(hist_base_ft['val_acc']) + 1)
ax.plot(ep2, hist_base_ft['val_acc'], label='Baseline (random init)', color='#94a3b8', lw=2)
ax.plot(ep2, hist_phys_ft['val_acc'], label='PhysREVE (pretrained)',  color='#2563eb', lw=2)
ax.axhline(0.25, ls='--', color='#ef4444', alpha=0.5, lw=1.2, label='Chance')
ax.set(title='Fine-tuning Validation Accuracy', xlabel='Epoch', ylabel='Accuracy')
ax.legend(fontsize=9); ax.grid(alpha=0.25); ax.set_ylim(0, 1)

# ── C: Test accuracy bar ──────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
models = ['Baseline\n(random init)', 'PhysREVE\n(physics pretrain)']
accs   = [acc_b, acc_p]
colors = ['#94a3b8', '#2563eb']
bars   = ax.bar(models, [a*100 for a in accs], color=colors, width=0.45, alpha=0.85)
ax.axhline(25, ls='--', color='#ef4444', alpha=0.6, lw=1.2, label='Chance (25%)')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{acc*100:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set(title='Test Accuracy', ylabel='Accuracy (%)', ylim=(0, 105))
ax.legend(fontsize=9); ax.grid(alpha=0.25, axis='y')

# ── D & E: Confusion matrices ─────────────────────────────────────────────────
for col, (preds, labs, title) in enumerate([
    (p_b, l_b, f'Baseline ({acc_b*100:.1f}%)'),
    (p_p, l_p, f'PhysREVE ({acc_p*100:.1f}%)')
], start=0):
    ax = fig.add_subplot(gs[1, col])
    cm_n = confusion_matrix(labs, preds, normalize='true')
    short = [c.split()[0] for c in CLASS_NAMES]
    sns.heatmap(cm_n, ax=ax, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=short, yticklabels=short, vmin=0, vmax=1, cbar=col==1)
    ax.set(title=title, xlabel='Predicted', ylabel='True')

# ── F: Source lateralization ──────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])

def motor_power_by_class(src_tensor, labels_arr, lh_i, rh_i):
    lh_pwr, rh_pwr = [], []
    for cls in range(N_CLASSES):
        mask = labels_arr == cls
        if mask.sum() > 0:
            lh_pwr.append(src_tensor[mask][:, lh_i, :].pow(2).mean().item())
            rh_pwr.append(src_tensor[mask][:, rh_i, :].pow(2).mean().item())
        else:
            lh_pwr.append(0.); rh_pwr.append(0.)
    return lh_pwr, rh_pwr

lh_p, rh_p = motor_power_by_class(s_p, l_p, lh_idx_np, rh_idx_np)

x_pos = np.arange(N_CLASSES); w = 0.35
ax.bar(x_pos-w/2, lh_p, w, label='LH motor (near C3)', color='#2563eb', alpha=0.8)
ax.bar(x_pos+w/2, rh_p, w, label='RH motor (near C4)', color='#dc2626', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels([c.split()[0] for c in CLASS_NAMES])
ax.set(title='Source Power by Class\n(PhysREVE)', ylabel='Mean Source Power (a.u.)')
ax.legend(fontsize=9); ax.grid(alpha=0.25, axis='y')
# Annotation for expected pattern
ax.text(0, max(lh_p[0],rh_p[0])*1.08, 'expect RH<LH', ha='center',
        fontsize=8, color='#15803d')
ax.text(1, max(lh_p[1],rh_p[1])*1.08, 'expect LH<RH', ha='center',
        fontsize=8, color='#15803d')

fig_path = os.path.join(EXP_DIR, f'figure_01_training_analysis{EXP_ID}.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig_path}')

In [ ]:
# ── Ablation Study ────────────────────────────────────────────────────────────
# Which physics component contributes what?
# We run three ablations on fine-tuning from the SAME pretrained encoder:
#   1. No physics fine-tuning losses (λ_phys=0, λ_asym=0)
#   2. Physics loss only (λ_phys>0, λ_asym=0)
#   3. Asymmetry loss only (λ_phys=0, λ_asym>0)
#   4. Both (full PhysREVE)

print('Running ablation study...')
ablation_results = {}

for name, lp, la in [
    ('No physics (CE only)',    0.0,             0.0),
    ('L_phys only',            CFG.lambda_phys,  0.0),
    ('L_asym only',            0.0,             CFG.lambda_asym),
    ('Full PhysREVE',          CFG.lambda_phys,  CFG.lambda_asym),
]:
    # Temporarily override config
    cfg_abl = PhysREVEConfig()
    cfg_abl.lambda_phys = lp
    cfg_abl.lambda_asym = la

    abl_model, _ = run_finetuning(
        pretrained_model, cfg_abl,
        finetune_loader, val_loader,
        n_epochs=40, lr_head=1e-3, lr_enc=3e-5
    )
    p_abl, l_abl, _ = evaluate(abl_model, test_loader)
    ablation_results[name] = (p_abl == l_abl).mean()
    print(f'  {name:<30}: {ablation_results[name]*100:.1f}%')

# ── Ablation bar chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
names = list(ablation_results.keys())
accs  = [ablation_results[n]*100 for n in names]
clrs  = ['#94a3b8', '#16a34a', '#9333ea', '#2563eb']
bars  = ax.barh(names, accs, color=clrs, alpha=0.85, height=0.5)
ax.axvline(25, ls='--', color='#ef4444', alpha=0.6, lw=1.2)
ax.axvline(acc_b*100, ls=':', color='#6b7280', alpha=0.7, lw=1.2,
           label=f'Baseline (random init): {acc_b*100:.1f}%')
for bar, acc in zip(bars, accs):
    ax.text(acc + 0.3, bar.get_y() + bar.get_height()/2,
            f'{acc:.1f}%', va='center', fontweight='bold')
ax.set(title='Ablation Study: Which Physics Loss Contributes What?\n'
             '(All use the same pretrained encoder)',
       xlabel='Test Accuracy (%)', xlim=(0, 100))
ax.legend(fontsize=9)
ax.grid(alpha=0.25, axis='x')
plt.tight_layout()
plt.savefig('physreve_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: physreve_ablation.png')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('\n' + '='*72)
print('PHYSREVE — COMPLETE SUMMARY')
print('='*72)
print(f"""
Architecture: PhysREVEEncoder
  4D pos enc    : {CFG.d_pos_x}+{CFG.d_pos_y}+{CFG.d_pos_z}+{CFG.d_pos_t} dims (x,y,z,t)  [REVE-identical]
  n_layers      : {CFG.n_layers}
  d_model       : {CFG.d_model}
  Leadfield bias: φ · B_ij added to attention  [NEW vs REVE]

Pretraining (unsupervised):
  Objective: L_MAE + {CFG.lambda_phys}·L_phys + {CFG.lambda_snr}·L_snr
  L_MAE:  Reconstruct masked patches (REVE)
  L_phys: ||norm(L @ source_est) - norm(sensor_patches)||²  [NEW]
  L_snr:  InfoNCE on high-SNR patch representations  [from EEGPT]

Fine-tuning (labeled BCI):
  Objective: L_CE + {CFG.lambda_phys}·L_phys + {CFG.lambda_asym}·L_asym
  L_asym: Contralateral ERD for left/right motor imagery  [NEW]

Forward model:
  Sphere conductor (MNE, no individual MRI needed)
  Leadfield L: {L_col.shape[0]} channels × {L_col.shape[1]} dipole sources
  Motor ROIs: {len(LH_IDX)} LH + {len(RH_IDX)} RH sources (near C3/C4)

Results (BCI IV 2a, Subject 1, 4-class MI):
  Baseline (random init)       : {acc_b*100:.1f}%
  PhysREVE (physics pretrained): {acc_p*100:.1f}%
  Improvement                  : {(acc_p-acc_b)*100:+.1f}%
  Chance level                 : 25.0%

Relationship to REVE (NeurIPS 2025):
  ✓  4D pos enc preserved exactly
  ✓  MAE objective preserved
  +  Source decoder branch added
  +  Leadfield physics constraint added
  +  SNR alignment loss added
  +  Hemispheric asymmetry loss for BCI fine-tuning

To scale to full REVE corpus (60k hrs, 92 datasets):
  → Replace DataLoader with multi-dataset iterator
  → Re-compute L per dataset (sphere model, ~30s per montage)
  → Use gradient accumulation for larger effective batch size
  → The physics losses are dataset-agnostic — no code changes needed
""")

# ── Experiment result log ────────────────────────────────────────────────────
_results = {
    'exp_id':      EXP_ID,
    'description': EXP_DESC,
    'timestamp':   datetime.now().isoformat(),
    'config': {
        'lambda_phys':  CFG.lambda_phys,
        'lambda_snr':   CFG.lambda_snr,
        'lambda_asym':  CFG.lambda_asym,
        'patch_size':   CFG.patch_size,
        'dropout':      CFG.dropout,
        'd_model':      CFG.d_model,
        'n_layers':     CFG.n_layers,
    },
    'results': {
        'baseline_acc': round(acc_b, 4),
        'physreve_acc': round(acc_p, 4),
        'improvement':  round(acc_p - acc_b, 4),
    },
}
_log_path = os.path.join(EXP_DIR, 'results.json')
with open(_log_path, 'w') as _f:
    _json.dump(_results, _f, indent=2)
print(f'Results logged: {_log_path}')


## 12. Scaling to Full REVE Corpus

To scale PhysREVE from this single-subject demo to REVE's 60k-hour, 92-dataset corpus:

### Multi-dataset dataloader
```python
class MultiDatasetEEGLoader:
    """
    Iterates over multiple EEG datasets, computing the leadfield
    on-the-fly for each dataset's electrode configuration.
    """
    def __init__(self, dataset_configs):
        self.datasets  = dataset_configs
        self.leadfields = {}   # cache L per montage

    def get_leadfield(self, ch_names):
        key = tuple(sorted(ch_names))
        if key not in self.leadfields:
            # Sphere model: ~30s per new montage
            L_col, L_row, src_pos, _ = build_leadfield(ch_names)
            self.leadfields[key] = (L_col, L_row, src_pos)
        return self.leadfields[key]

    def __iter__(self):
        for ds in self.datasets:
            L_col, L_row, _ = self.get_leadfield(ds.ch_names)
            for batch in ds.dataloader:
                yield batch, L_col, L_row   # pass dataset-specific L
```

### Key point
The physics losses are **dataset-agnostic** — the `physics_loss` and `snr_alignment_loss`
functions work identically regardless of the number of channels or electrode layout.
Only `L` changes per dataset, which is precomputed and cached.

### Memory
With 92 montages, each L ≈ (64 channels × 1284 sources × 4 bytes) ≈ 330 KB.  
Total leadfield cache: 92 × 330 KB ≈ 30 MB — negligible.

### The motor asymmetry loss at scale
During pretraining on unlabeled data, `L_asym` is **not** used — it requires labels.  
At REVE scale, only `L_MAE + L_phys + L_snr` are used in pretraining.  
`L_asym` is added only during BCI fine-tuning, exactly as in this notebook.

### Expected gains vs REVE
The largest expected improvement is on **motor imagery** (currently 0.534 B-Acc in REVE),
where the asymmetry loss provides a strong, cross-subject-consistent supervision signal.
Sleep staging and seizure detection will benefit mainly from `L_phys + L_snr`
forcing the encoder to learn anatomically grounded representations during pretraining.

# PhysREVE Paper Outline

## 1. Abstract
- Introduce PhysREVE as a physics-informed EEG foundation model.
- Explain the two core contributions:
  - `L_snr`: an EEGPT-inspired SNR alignment loss for source-space consistency.
  - `L_phys`: a DeepSIF-inspired leadfield reconstruction loss tying decoded sources back to sensor measurements.
- Mention the dataset: BCI Competition IV 2a, Subject 1, 4-class motor imagery.
- Summarize the experiment story: initial prototype diagnostics, hyperparameter fixes, and the promise of physics-guided EEG representation learning.

## 2. Introduction
- Motivation: most EEG foundation models learn only from scalp signals and ignore the forward physics that generated them.
- Claim: embedding source-space physics into self-supervised pretraining improves representation fidelity and transfer.
- High-level problem statement: `y = L s + ε` and the challenge of learning from scalp EEG alone.
- Contribution summary and expected impact.

## 3. Related Work
- EEG foundation models and transformer-based EEG pretraining (REVE, EEGPT, others).
- Physics-informed EEG/source localization methods (DeepSIF, MNE forward modelling, inverse problem regularization).
- Contrastive and SNR-based self-supervised losses in EEG.

## 4. Method
- Model overview: EEG patch tokenizer, 4D positional encoding, transformer backbone, source decoder, forward projection.
- Loss components:
  - `L_mae`: masked reconstruction loss.
  - `L_phys`: leadfield consistency loss `||L·ŝ - y||^2`.
  - `L_snr`: high-SNR patch alignment loss.
  - `L_asym`: motor imagery hemispheric ERD asymmetry prior.
- Training protocol:
  - Phase 1: self-supervised pretraining.
  - Phase 2: fine-tuning on 4-class motor imagery.

## 5. Experiments
- Dataset and experimental setup.
- Baseline and PhysREVE configurations.
- Experiment log: `EXP_001`, `EXP_002`, ...
- Evaluation metrics and ablation structure.

## 6. Results and Analysis
- Quantitative results: baseline vs PhysREVE, progress over experiments.
- Diagnostic analysis: dead SNR loss, asymmetry saturation, same-subject pretraining, patch geometry issues.
- What worked and what needs improvement.

## 7. Discussion
- Interpretation of the physics-guided approach.
- Practical lessons for EEG self-supervised learning.
- Limitations of the current prototype.
- Roadmap for stronger cross-subject pretraining and 3D source-to-scalp validation.

## 8. Conclusion
- Restate the value of combining EEGPT-inspired SNR loss with DeepSIF-inspired leadfield loss.
- Emphasize that physics-informed EEG pretraining is promising but requires careful alignment of loss design and patching strategy.
- Outline next steps for a stronger paper.

## 9. Figure Plan
- Figure 1: PhysREVE architecture schematic.
- Figure 2: 3D brain-to-scalp source-to-sensor visualization.
- Figure 3: Experiment progress plot comparing `EXP_001` and `EXP_002`.
- Figure 4: Bar chart/table of accuracy and improvement.
- Figure 5: Source asymmetry maps for left/right motor imagery.
